# 03 — Regularization and Generalization

Learn how dropout and weight decay reduce overfitting.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(10)

In [ ]:
X = torch.randn(1800, 30)
y = ((X[:, :3].sum(dim=1) + 0.2*torch.randn(1800)) > 0).long()
dataset = TensorDataset(X, y)
train_ds, val_ds = random_split(dataset, [1400, 400])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

In [ ]:
def fit(dropout=0.0, weight_decay=0.0):
    model = nn.Sequential(
        nn.Linear(30, 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(64, 2)
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    for _ in range(25):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        tr_acc = (model(X[train_ds.indices]).argmax(1) == y[train_ds.indices]).float().mean().item()
        va_acc = (model(X[val_ds.indices]).argmax(1) == y[val_ds.indices]).float().mean().item()
    return tr_acc, va_acc

base = fit(dropout=0.0, weight_decay=0.0)
reg = fit(dropout=0.3, weight_decay=1e-4)

print('No regularization  -> train, val:', [round(v, 4) for v in base])
print('With regularization-> train, val:', [round(v, 4) for v in reg])

## Exercises
1. Tune dropout from 0.1 to 0.5.
2. Add early stopping based on validation loss.